In [1]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

!pip install opencv-python-headless seaborn -q

Mounted at /content/drive


In [3]:
from pathlib import Path
import json

# load FV40 dataset and annotations
FV40_DIR = Path("/content/drive/Shareddrives/Computer Vision Final Project/Data/fv40_merged")
with open(FV40_DIR / "annotations" / "instances.json") as f:
    coco = json.load(f)

cat_map = {c["id"]: c["name"] for c in coco["categories"]}

In [4]:
# Additional setup for the FOOD-INGREDIENTS dataset (YOLOv7 PyTorch / Roboflow export)
!pip install pyyaml -q

import yaml

# Assumes the dataset sits alongside fv40_merged inside the same "Data" folder, i.e.
# .../Data/FOOD-INGREDIENTS Dataset/  ->  update this if your Drive layout differs.
FOOD_DIR = FV40_DIR.parent / "food_ingredients"

print("Resolved FOOD_DIR:", FOOD_DIR)
assert (FOOD_DIR / "data.yaml").exists(), (
    f"data.yaml not found at {FOOD_DIR} - update FOOD_DIR to match your Drive layout."
)

with open(FOOD_DIR / "data.yaml") as f:
    food_yaml = yaml.safe_load(f)

names = food_yaml["names"]
food_cat_map = {int(k): v for k, v in names.items()} if isinstance(names, dict) else dict(enumerate(names))


Resolved FOOD_DIR: /content/drive/Shareddrives/Computer Vision Final Project/Data/food_ingredients


In [5]:
# Retrieve both class lists for building the merge mapping
try:
    fv40_names = sorted(cat_map.values())
    food_names = sorted(food_cat_map.values())
except NameError:
    import json
    from pathlib import Path
    import yaml

    FV40_DIR = Path("/content/drive/Shareddrives/Computer Vision Final Project/Data/fv40_merged")
    FOOD_DIR = FV40_DIR.parent / "food_ingredients"

    with open(FV40_DIR / "annotations" / "instances.json") as f:
        coco = json.load(f)
    fv40_names = sorted(c["name"] for c in coco["categories"])

    with open(FOOD_DIR / "data.yaml") as f:
        data_yaml = yaml.safe_load(f)
    food_names = sorted(data_yaml["names"])

print(f"FV40 ({len(fv40_names)} classes):")
print(fv40_names)
print(f"\nFOOD-INGREDIENTS ({len(food_names)} classes):")
print(food_names)


FV40 (62 classes):
['almond', 'apple', 'apricot', 'artichoke', 'asparagus', 'avocado', 'banana', 'bean curd', 'bell pepper', 'blackberry', 'blueberry', 'broccoli', 'brussels sprouts', 'cantaloup', 'carrot', 'cauliflower', 'cayenne', 'celery', 'cherry', 'chickpea', 'chili', 'citrus', 'clementine', 'coconut', 'cucumber', 'date', 'edible corn', 'eggplant', 'fig', 'garlic', 'ginger', 'gourd', 'grape', 'green bean', 'green onion', 'kiwi fruit', 'lemon', 'lettuce', 'lime', 'mandarin orange', 'melon', 'mushroom', 'onion', 'orange', 'papaya', 'pea', 'peach', 'pear', 'persimmon', 'pickle', 'pineapple', 'potato', 'prune', 'pumpkin', 'radish', 'raspberry', 'strawberry', 'sweet potato', 'tomato', 'turnip', 'watermelon', 'zucchini']

FOOD-INGREDIENTS (120 classes):
['Akabare Khursani', 'Apple', 'Artichoke', 'Ash Gourd -Kubhindo-', 'Asparagus -Kurilo-', 'Avocado', 'Bacon', 'Bamboo Shoots -Tama-', 'Banana', 'Beans', 'Beaten Rice -Chiura-', 'Beef', 'Beetroot', 'Bethu ko Saag', 'Bitter Gourd', 'Black L

# Merging FV40 and FOOD-INGREDIENTS

The two class lists above share zero exact string matches, but many classes are the same
ingredient under a different name (`bell pepper` / `Capsicum`, `eggplant` / `Brinjal`) or with
extra Roboflow/Nepali-transliteration decoration (`Asparagus -Kurilo-`). The approach:

1. Auto-generate candidate synonym pairs (normalization + fuzzy matching) as an audit trail.
2. Hand-review those candidates into a **confirmed mapping** — only true synonyms get merged;
   everything else (including near-misses that turned out to be genuinely different, like
   `cayenne` vs the already-claimed `Chili Pepper -Khursani-`) stays as its own class.
3. Build one unified canonical taxonomy from FV40's 62 + FOOD-INGREDIENTS' 120 classes.
4. Remap both datasets' annotations onto the unified taxonomy and merge into a single COCO-style
   `instances.json`, tagging which classes each source dataset actually had annotations for
   (needed later for partial-label loss masking — a source's images aren't confirmed negatives
   for classes only the *other* dataset annotated).

In [6]:
import re
from difflib import SequenceMatcher
import pandas as pd
import numpy as np

def clean_name(name):
    """Lowercase, strip Roboflow's ' -Nepali transliteration-' suffixes, collapse whitespace."""
    name = re.sub(r"-[^-]+-", " ", name)
    name = re.sub(r"\s+", " ", name).strip().lower()
    return name

def similarity(a, b):
    return SequenceMatcher(None, clean_name(a), clean_name(b)).ratio()

# Candidate pairs: for each FV40 class, the closest FOOD-INGREDIENTS class by normalized
# string similarity. This is a suggestion list ONLY, meant to be eyeballed below -
# nothing here is applied automatically.
candidates = []
for fv in fv40_names:
    best_food, best_score = max(
        ((food, similarity(fv, food)) for food in food_names),
        key=lambda t: t[1],
    )
    candidates.append((fv, best_food, round(best_score, 2)))

candidates_df = pd.DataFrame(candidates, columns=["fv40_class", "closest_food_class", "similarity"])
candidates_df = candidates_df.sort_values("similarity", ascending=False)
print(candidates_df.to_string(index=False))

      fv40_class              closest_food_class  similarity
           apple                           Apple        1.00
       artichoke                       Artichoke        1.00
         avocado                         Avocado        1.00
       asparagus              Asparagus -Kurilo-        1.00
        broccoli                        Broccoli        1.00
          banana                          Banana        1.00
        cucumber                        Cucumber        1.00
          garlic                          Garlic        1.00
     cauliflower                     Cauliflower        1.00
          carrot                          Carrot        1.00
          papaya                          Papaya        1.00
             pea                             Pea        1.00
           lemon                   Lemon -Nimbu-        1.00
          ginger                          Ginger        1.00
            pear                            Pear        1.00
          potato        

## Confirmed mapping

Fuzzy similarity gets close but isn't trustworthy on its own here — e.g. it'll happily suggest
`cayenne` ~ `Chili Pepper -Khursani-`, but that FOOD-INGREDIENTS class is already the correct
match for FV40's `chili`, and cayenne is a distinct-enough cultivar that forcing it into the same
bucket would conflate two FV40 classes into one. The table below is the hand-reviewed result:
only pairs I'm confident name the same ingredient get merged. Everything else — including the
near-misses — is listed explicitly with the reason it was left alone, so the decision is
auditable rather than silent.

In [7]:
# (fv40_class, food_class, canonical_name) - confirmed synonym pairs to merge
CONFIRMED_PAIRS = [
    ("apple", "Apple", "apple"),
    ("artichoke", "Artichoke", "artichoke"),
    ("asparagus", "Asparagus -Kurilo-", "asparagus"),
    ("avocado", "Avocado", "avocado"),
    ("banana", "Banana", "banana"),
    ("bean curd", "Tofu", "tofu"),
    ("bell pepper", "Capsicum", "bell pepper"),
    ("broccoli", "Broccoli", "broccoli"),
    ("carrot", "Carrot", "carrot"),
    ("cauliflower", "Cauliflower", "cauliflower"),
    ("chickpea", "Chickpeas", "chickpea"),
    ("chili", "Chili Pepper -Khursani-", "chili pepper"),
    ("cucumber", "Cucumber", "cucumber"),
    ("edible corn", "Corn", "corn"),
    ("eggplant", "Brinjal", "eggplant"),
    ("garlic", "Garlic", "garlic"),
    ("ginger", "Ginger", "ginger"),
    ("green onion", "Onion Leaves", "green onion"),
    ("lemon", "Lemon -Nimbu-", "lemon"),
    ("lime", "Lime -Kagati-", "lime"),
    ("mushroom", "Mushroom", "mushroom"),
    ("onion", "Onion", "onion"),
    ("orange", "Orange", "orange"),
    ("papaya", "Papaya", "papaya"),
    ("pea", "Pea", "pea"),
    ("pear", "Pear", "pear"),
    ("potato", "Potato", "potato"),
    ("pumpkin", "Pumpkin -Farsi-", "pumpkin"),
    ("radish", "Radish", "radish"),
    ("strawberry", "Strawberry", "strawberry"),
    ("sweet potato", "Sweet Potato -Suthuni-", "sweet potato"),
    ("tomato", "Tomato", "tomato"),
    ("turnip", "Turnip", "turnip"),
    ("watermelon", "Water Melon", "watermelon"),
]

# Near-misses that fuzzy matching would suggest, deliberately NOT merged - kept for
# whoever reviews this next so the choice doesn't have to be re-derived.
REVIEWED_NOT_MERGED = [
    ("cayenne", "Chili Pepper -Khursani- / Akabare Khursani",
     "distinct pepper cultivar; the FOOD chili class is already claimed by FV40's 'chili'"),
    ("citrus", None,
     "FV40's generic catch-all; FOOD only has specific citrus fruits, already claimed by their own FV40 matches"),
    ("clementine", None, "no FOOD-INGREDIENTS equivalent"),
    ("mandarin orange", "Orange",
     "FV40 already distinguishes this from plain 'orange'; FOOD's one 'Orange' class is already claimed"),
    ("gourd", "Ash Gourd / Bitter Gourd / Bottle Gourd / Snake Gourd / Sponge Gourd / Pointed Gourd",
     "FV40's generic bucket has no 1:1 counterpart among FOOD's specific gourd varieties"),
    ("green bean", "Beans / Broad Beans -Bakullo- / Long Beans -Bodi-",
     "ambiguous which FOOD class corresponds; left unmerged"),
    ("melon / cantaloup", None, "no FOOD-INGREDIENTS equivalent"),
]

print(f"{len(CONFIRMED_PAIRS)} classes merged as synonyms.")
print(f"{len(REVIEWED_NOT_MERGED)} near-misses reviewed and deliberately kept separate:")
for fv, food, reason in REVIEWED_NOT_MERGED:
    print(f"  - {fv}  (closest FOOD candidate: {food}) -> {reason}")

34 classes merged as synonyms.
7 near-misses reviewed and deliberately kept separate:
  - cayenne  (closest FOOD candidate: Chili Pepper -Khursani- / Akabare Khursani) -> distinct pepper cultivar; the FOOD chili class is already claimed by FV40's 'chili'
  - citrus  (closest FOOD candidate: None) -> FV40's generic catch-all; FOOD only has specific citrus fruits, already claimed by their own FV40 matches
  - clementine  (closest FOOD candidate: None) -> no FOOD-INGREDIENTS equivalent
  - mandarin orange  (closest FOOD candidate: Orange) -> FV40 already distinguishes this from plain 'orange'; FOOD's one 'Orange' class is already claimed
  - gourd  (closest FOOD candidate: Ash Gourd / Bitter Gourd / Bottle Gourd / Snake Gourd / Sponge Gourd / Pointed Gourd) -> FV40's generic bucket has no 1:1 counterpart among FOOD's specific gourd varieties
  - green bean  (closest FOOD candidate: Beans / Broad Beans -Bakullo- / Long Beans -Bodi-) -> ambiguous which FOOD class corresponds; left unmerged


In [8]:
# Build the two raw-name -> canonical-name lookups. Classes with no confirmed synonym just
# map to their own cleaned name, so nothing gets dropped - see the question this answers:
# unique classes are kept, only true duplicates are collapsed.
FV40_TO_CANONICAL = {fv: canon for fv, food, canon in CONFIRMED_PAIRS}
FOOD_TO_CANONICAL = {food: canon for fv, food, canon in CONFIRMED_PAIRS}

for name in fv40_names:
    FV40_TO_CANONICAL.setdefault(name, clean_name(name))
for name in food_names:
    FOOD_TO_CANONICAL.setdefault(name, clean_name(name))

assert len(FV40_TO_CANONICAL) == len(fv40_names)
assert len(FOOD_TO_CANONICAL) == len(food_names)

# Sanity check: make sure two *unmerged* classes didn't accidentally clean down to the same
# canonical name (that would silently merge classes we never reviewed as synonyms).
canon_to_sources = {}
for name, canon in {**FV40_TO_CANONICAL, **{f"FOOD:{k}": v for k, v in FOOD_TO_CANONICAL.items()}}.items():
    canon_to_sources.setdefault(canon, []).append(name)
unreviewed_collisions = {
    canon: sources for canon, sources in canon_to_sources.items()
    if len(sources) > 1 and canon not in {c for _, _, c in CONFIRMED_PAIRS}
}
assert not unreviewed_collisions, f"Unreviewed name collisions found: {unreviewed_collisions}"

canonical_classes = sorted(set(FV40_TO_CANONICAL.values()) | set(FOOD_TO_CANONICAL.values()))
CANONICAL_TO_ID = {name: i for i, name in enumerate(canonical_classes)}

print(f"FV40: {len(fv40_names)} classes, FOOD-INGREDIENTS: {len(food_names)} classes")
print(f"Naive union (no merging): {len(set(fv40_names) | set(food_names))} classes")
print(f"Unified taxonomy (after merging {len(CONFIRMED_PAIRS)} synonym pairs): {len(canonical_classes)} classes")

FV40: 62 classes, FOOD-INGREDIENTS: 120 classes
Naive union (no merging): 182 classes
Unified taxonomy (after merging 34 synonym pairs): 148 classes


## Loading full annotations from both datasets

Same parsing logic as the EDA notebook (COCO json for FV40, YOLO txt files for
FOOD-INGREDIENTS), but with `category_id` remapped through the canonical taxonomy instead of
each dataset's own class ids.

In [9]:
# FV40: images + annotations, category_id remapped to the unified canonical taxonomy
fv40_images_df = pd.DataFrame(coco["images"])
fv40_anns_df = pd.DataFrame(coco["annotations"])

fv40_anns_df["canonical_name"] = (
    fv40_anns_df["category_id"].map(cat_map).map(FV40_TO_CANONICAL)
)
fv40_anns_df["canonical_id"] = fv40_anns_df["canonical_name"].map(CANONICAL_TO_ID)

print(f"FV40: {len(fv40_images_df):,} images, {len(fv40_anns_df):,} boxes")
assert fv40_anns_df["canonical_id"].isna().sum() == 0

FV40: 17,459 images, 282,794 boxes


In [10]:
# FOOD-INGREDIENTS: parse YOLO txt labels, convert normalized center-boxes to COCO-style
# pixel [x_min, y_min, w, h], category_id remapped to the unified canonical taxonomy
DEFAULT_W, DEFAULT_H = 640, 640
SPLITS = ["train", "valid", "test"]

food_image_rows, food_ann_rows = [], []
img_id, ann_id = 0, 0

for split in SPLITS:
    images_dir = FOOD_DIR / split / "images"
    labels_dir = FOOD_DIR / split / "labels"
    if not images_dir.exists():
        continue
    for img_path in sorted(images_dir.iterdir()):
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        food_image_rows.append({
            "id": img_id, "file_name": img_path.name,
            "width": DEFAULT_W, "height": DEFAULT_H, "split": split,
        })
        label_path = labels_dir / (img_path.stem + ".txt")
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                food_name = food_cat_map.get(cls_id, f"class_{cls_id}")
                canonical_name = FOOD_TO_CANONICAL[food_name]
                box_w, box_h = bw * DEFAULT_W, bh * DEFAULT_H
                x_min, y_min = xc * DEFAULT_W - box_w / 2, yc * DEFAULT_H - box_h / 2
                food_ann_rows.append({
                    "id": ann_id, "image_id": img_id,
                    "canonical_name": canonical_name,
                    "canonical_id": CANONICAL_TO_ID[canonical_name],
                    "bbox": [x_min, y_min, box_w, box_h],
                    "area": box_w * box_h,
                    "split": split,
                })
                ann_id += 1
        img_id += 1

food_images_df = pd.DataFrame(food_image_rows)
food_anns_df = pd.DataFrame(food_ann_rows)

print(f"FOOD-INGREDIENTS: {len(food_images_df):,} images, {len(food_anns_df):,} boxes")

FOOD-INGREDIENTS: 9,734 images, 22,843 boxes


## Merging into a single unified COCO-style dataset

Images keep a `source_dataset` tag and a resolvable `file_path` rather than being copied. Each
image gets a new global id; annotations are re-pointed at those ids and at the canonical
`category_id`.

Also recorded: `source_class_coverage` - which canonical classes each source dataset actually
had in its own taxonomy. This matters because a FV40 image with no `eggplant` box is *not*
necessarily proof the image has no eggplant in it if `eggplant` was only ever annotated in
FOOD-INGREDIENTS - the two datasets should not be treated as mutual confirmed-negatives outside
their `CONFIRMED_PAIRS` overlap. Worth handling with a masked loss when training on the merged
set (skip loss for canonical classes outside the source image's own dataset's coverage), rather
than pretending every unannotated class-in-image is a true negative.

In [11]:
merged_images = []
merged_annotations = []
next_image_id = 0
next_ann_id = 0

# --- FV40 images/annotations ---
fv40_id_remap = {}
for _, row in fv40_images_df.iterrows():
    fv40_id_remap[row["id"]] = next_image_id
    merged_images.append({
        "id": next_image_id,
        "file_path": str(FV40_DIR / "images" / row["file_name"]),
        "width": row["width"],
        "height": row["height"],
        "source_dataset": "fv40",
        "split": None,
    })
    next_image_id += 1

for _, row in fv40_anns_df.iterrows():
    merged_annotations.append({
        "id": next_ann_id,
        "image_id": fv40_id_remap[row["image_id"]],
        "category_id": int(row["canonical_id"]),
        "bbox": row["bbox"],
        "area": row["bbox"][2] * row["bbox"][3],
        "source_dataset": "fv40",
    })
    next_ann_id += 1

# --- FOOD-INGREDIENTS images/annotations ---
food_id_remap = {}
for _, row in food_images_df.iterrows():
    food_id_remap[row["id"]] = next_image_id
    merged_images.append({
        "id": next_image_id,
        "file_path": str(FOOD_DIR / row["split"] / "images" / row["file_name"]),
        "width": row["width"],
        "height": row["height"],
        "source_dataset": "food_ingredients",
        "split": row["split"],
    })
    next_image_id += 1

for _, row in food_anns_df.iterrows():
    merged_annotations.append({
        "id": next_ann_id,
        "image_id": food_id_remap[row["image_id"]],
        "category_id": int(row["canonical_id"]),
        "bbox": row["bbox"],
        "area": row["area"],
        "source_dataset": "food_ingredients",
    })
    next_ann_id += 1

merged_categories = [{"id": cid, "name": name} for name, cid in sorted(CANONICAL_TO_ID.items(), key=lambda t: t[1])]

source_class_coverage = {
    "fv40": sorted({CANONICAL_TO_ID[c] for c in FV40_TO_CANONICAL.values()}),
    "food_ingredients": sorted({CANONICAL_TO_ID[c] for c in FOOD_TO_CANONICAL.values()}),
}

merged_dataset = {
    "images": merged_images,
    "annotations": merged_annotations,
    "categories": merged_categories,
    "source_class_coverage": source_class_coverage,
}

print(f"Merged: {len(merged_images):,} images, {len(merged_annotations):,} boxes, {len(merged_categories)} classes")
print(f"  from FV40: {len(fv40_images_df):,} images, {len(fv40_anns_df):,} boxes")
print(f"  from FOOD-INGREDIENTS: {len(food_images_df):,} images, {len(food_anns_df):,} boxes")

Merged: 27,193 images, 305,637 boxes, 148 classes
  from FV40: 17,459 images, 282,794 boxes
  from FOOD-INGREDIENTS: 9,734 images, 22,843 boxes


In [12]:
MERGED_DIR = FV40_DIR.parent / "merged"
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_path = MERGED_DIR / "instances.json"

with open(merged_path, "w") as f:
    json.dump(merged_dataset, f)

print(f"Wrote merged dataset to {merged_path}")

# Per-class box counts in the merged set, so imbalance across the combined taxonomy is visible
merged_class_counts = (
    pd.DataFrame(merged_annotations)["category_id"]
    .map({c["id"]: c["name"] for c in merged_categories})
    .value_counts()
)
print(f"\nTop 10 classes:\n{merged_class_counts.head(10)}")
print(f"\nBottom 10 classes:\n{merged_class_counts.tail(10)}")

Wrote merged dataset to /content/drive/Shareddrives/Computer Vision Final Project/Data/merged/instances.json

Top 10 classes:
category_id
banana      54365
citrus      42653
orange      28613
apple       24526
carrot      18340
grape       13676
tomato      12800
broccoli    12295
onion       10132
mushroom     7264
Name: count, dtype: int64

Bottom 10 classes:
category_id
olive oil         18
black beans       16
green peas        14
crab meat         10
ice                9
wallnut            9
yellow lentils     8
long beans         6
fish               6
wheat              1
Name: count, dtype: int64
